# Paso 4 - Analisis cualitativo y tradeoffs de novedad/diversidad

Este notebook responde tres preguntas practicas para el paper/poster:

1. Que metrica esta optimizando realmente el modelo.
2. Si novedad/diversidad solo se miden o si tambien se pueden promover.
3. Que se le recomienda a usuarios concretos, que caracteristicas tienen y por que el modelo cambia el ranking.

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path('..').resolve().parent
SRC_DIR = PROJECT_DIR / 'H3' / 'src'
OUTPUT_DIR = PROJECT_DIR / 'H3' / 'outputs'
sys.path.insert(0, str(SRC_DIR))

from step2_tune_adaptive import (
    cat_recs,
    combined_recs,
    hier_recs,
    load_core,
    seq_recs,
    transition_confidence,
    weights_for,
)
from step1_metrics_examples import (
    category_diversity_at_k,
    filter_seen,
    novelty_at_k,
    ranking_metrics,
)

K = 10
MAX_EVAL_USERS = 50_000
SEED = 42

pd.set_option('display.max_colwidth', 180)

## 1. Metrica objetivo del modelo

El tuning del modelo se hizo usando **Recall@10 como criterio principal** y **nDCG@10 como criterio secundario**. Es decir, el modelo fue seleccionado por su capacidad de incluir el item real de test dentro del top-10, y luego por ubicarlo mejor en el ranking.

Las metricas de novedad, diversidad y coverage se usaron inicialmente como metricas de diagnostico. Sirven para evaluar el tradeoff, pero no eran parte directa de la funcion de seleccion del modelo tuneado.

Por eso agregamos abajo un experimento adicional: un **reranking post-hoc** que si promueve novedad/diversidad explicitamente. Esto permite mostrar que se pueden mejorar esas dimensiones, pero con posible costo en Recall/nDCG.

In [2]:
final_summary = pd.read_csv(OUTPUT_DIR / 'final_metrics_summary_full.csv')
final_short = pd.read_csv(OUTPUT_DIR / 'final_metrics_short_history_full.csv')
tuning = pd.read_csv(OUTPUT_DIR / 'step2_tuning_results_sample_50000.csv')

display(final_summary)
display(final_short)
display(tuning.sort_values(['recall@10', 'ndcg@10'], ascending=False).head(10))

,model,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10,catalog_coverage@10
0,Adaptive Hierarchical Hybrid (tuned),0.011413,0.114127,0.069032,13.596369,0.505077,0.495391
1,Fixed Hybrid Seq+Category,0.010945,0.109453,0.068414,13.165151,0.621789,0.494210
2,Sequential Transition,0.010251,0.102515,0.066455,13.245382,0.728481,0.516679
3,ItemKNN,0.007913,0.079134,0.043888,15.419379,0.741596,0.638723
4,Category Popularity,0.007632,0.076324,0.040498,13.500720,0.070065,0.078255
5,Most Popular,0.000219,0.002194,0.000931,10.198046,0.955455,0.000265
6,Random,0.000005,0.000052,0.000021,18.787670,0.995093,1.000000


,model,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10
0,Adaptive Hierarchical Hybrid (tuned),0.012759,0.127589,0.077486,13.524790,0.500161
1,Fixed Hybrid Seq+Category,0.012192,0.121917,0.076885,13.052526,0.638767
2,Sequential Transition,0.011390,0.113902,0.074636,13.095803,0.741881
3,ItemKNN,0.008757,0.087572,0.049372,15.090811,0.754958
4,Category Popularity,0.008523,0.085226,0.045413,13.454335,0.089281
5,Most Popular,0.000239,0.002391,0.001000,10.197705,0.955508
6,Random,0.000005,0.000053,0.000024,18.787995,0.995088


,name,seq_min,seq_span,cat_base,cat_drop,hier_max,hier_power,pop,transition_mix,use_hierarchy,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10,short_recall@10,short_ndcg@10,catalog_coverage@10
0,smin0.15_sspan0.7_h0.2,0.15,0.7,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011320,0.11320,0.068543,13.598209,0.506618,0.125735,0.076627,0.299967
1,smin0.15_sspan0.6_h0.2,0.15,0.6,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011320,0.11320,0.067986,13.584909,0.489448,0.125907,0.076048,0.295484
2,smin0.15_sspan0.7_h0.16,0.15,0.7,0.55,0.3,0.16,1.0,0.05,0.75,True,0.011296,0.11296,0.068654,13.580007,0.513529,0.125535,0.076810,0.300438
3,smin0.15_sspan0.6_h0.16,0.15,0.6,0.55,0.3,0.16,1.0,0.05,0.75,True,0.011296,0.11296,0.068029,13.562571,0.497613,0.125535,0.076098,0.295623
4,smin0.15_sspan0.5_h0.2,0.15,0.5,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011296,0.11296,0.067296,13.558787,0.473344,0.125650,0.075259,0.289257
5,smin0.15_sspan0.5_h0.16,0.15,0.5,0.55,0.3,0.16,1.0,0.05,0.75,True,0.011290,0.11290,0.067423,13.540838,0.480558,0.125621,0.075414,0.289681
6,smin0.2_sspan0.6_h0.2,0.20,0.6,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011282,0.11282,0.068457,13.553310,0.514346,0.125307,0.076578,0.298163
7,smin0.2_sspan0.7_h0.2,0.20,0.7,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011280,0.11280,0.068747,13.570847,0.527615,0.125336,0.076930,0.302434
8,smin0.25_sspan0.7_h0.2,0.25,0.7,0.55,0.3,0.20,1.0,0.05,0.75,True,0.011276,0.11276,0.069078,13.570896,0.538251,0.125393,0.077382,0.304649
9,smin0.2_sspan0.7_h0.12,0.20,0.7,0.55,0.3,0.12,1.0,0.05,0.75,True,0.011274,0.11274,0.068965,13.549543,0.536161,0.125393,0.077252,0.303097


## 2. Cargar artefactos y reconstruir componentes del modelo

Usamos los caches ya generados para evitar recalcular split, categorias, transiciones y rankings jerarquicos.

In [3]:
core = load_core(PROJECT_DIR, max_eval_users=MAX_EVAL_USERS, seed=SEED)
config = json.loads((OUTPUT_DIR / 'step2_best_config_sample_50000.json').read_text())
test_item = core['test'].set_index('visitorid')['itemid'].astype(int).to_dict()

print(f'Usuarios evaluados en este notebook: {len(core["eval_users"]):,}')
print(config)

Usuarios evaluados en este notebook: 50,000
{'name': 'smin0.15_sspan0.7_h0.2', 'seq_min': 0.15, 'seq_span': 0.7, 'cat_base': 0.55, 'cat_drop': 0.3, 'hier_max': 0.2, 'hier_power': 1.0, 'pop': 0.05, 'transition_mix': 0.75, 'use_hierarchy': True}


In [4]:
def build_components_for_user(user, n=200):
    return {
        user: {
            'seq': seq_recs(user, core, n),
            'cat': cat_recs(user, core, n),
            'hier': hier_recs(user, core, n),
        }
    }


def fixed_hybrid_recs(user, components, k=10):
    scores = defaultdict(float)
    for rank, item in enumerate(components[user]['seq'][:200], start=1):
        scores[int(item)] += 0.70 / rank
    for rank, item in enumerate(components[user]['cat'][:200], start=1):
        scores[int(item)] += 0.30 / rank
    for rank, item in enumerate(core['fallback_items'][:200], start=1):
        scores[int(item)] += 0.05 / rank
    ranked = [item for item, _ in sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))]
    return filter_seen(ranked + core['fallback_items'], core['seen_items'].get(user, set()), k)


def tuned_recs(user, components, k=10):
    return combined_recs(user, core, components, config, k, max_rank=200)


def user_profile(user):
    user_train = core['train'][core['train']['visitorid'].eq(user)].sort_values('timestamp')
    history = []
    for row in user_train[['event', 'itemid', 'event_weight']].tail(8).itertuples(index=False):
        item = int(row.itemid)
        history.append({
            'event': row.event,
            'itemid': item,
            'categoryid': core['item_to_category'].get(item),
            'weight': row.event_weight,
        })
    t_conf = transition_confidence(user, core)
    h_len = int(core['train_history_lengths'].get(user, 0))
    weights = weights_for(config, t_conf, h_len)
    return history, t_conf, h_len, weights

## 3. Reranking para promover novedad/diversidad

El modelo final tuneado promueve principalmente relevancia, medida por Recall@10 y nDCG@10. Para promover novedad/diversidad de manera explicita, probamos un reranking sobre los candidatos del modelo final.

La idea es simple: partimos del ranking del modelo final y reordenamos candidatos con una mezcla de:

- score de relevancia aproximado por posicion original,
- bonus de novedad: items menos populares reciben mas puntaje,
- bonus de diversidad: se favorecen categorias aun no representadas en la lista parcial.

Esto no reemplaza al modelo principal; sirve como experimento de tradeoff.

In [5]:
def rerank_novelty_diversity(user, base_candidates, k=10, relevance_weight=1.0, novelty_weight=0.0, diversity_weight=0.0):
    selected = []
    selected_categories = set()
    candidates = list(dict.fromkeys(int(item) for item in base_candidates))
    max_novelty = math.log2(max(core['catalog_size'], 2))

    while candidates and len(selected) < k:
        best_item = None
        best_score = -1e18
        for rank, item in enumerate(candidates, start=1):
            relevance = 1.0 / math.log2(rank + 1)
            novelty = novelty_at_k([item], core['item_prob'], core['catalog_size'], 1) / max_novelty
            category = core['item_to_category'].get(item)
            diversity_bonus = 1.0 if category is not None and category not in selected_categories else 0.0
            score = relevance_weight * relevance + novelty_weight * novelty + diversity_weight * diversity_bonus
            if score > best_score:
                best_item = item
                best_score = score
        selected.append(best_item)
        category = core['item_to_category'].get(best_item)
        if category is not None:
            selected_categories.add(category)
        candidates.remove(best_item)
    return selected


def evaluate_recs_for_users(recs_by_user):
    rows = []
    coverage = set()
    for user, recs in recs_by_user.items():
        true_item = int(test_item[user])
        coverage.update(recs)
        row = ranking_metrics(recs, true_item, K)
        row[f'novelty@{K}'] = novelty_at_k(recs, core['item_prob'], core['catalog_size'], K)
        row[f'category_diversity@{K}'] = category_diversity_at_k(recs, core['item_to_category'], K)
        row['history_length'] = int(core['train_history_lengths'].get(user, 0))
        rows.append(row)
    df = pd.DataFrame(rows)
    summary = {col: df[col].mean() for col in df.columns if col != 'history_length'}
    summary[f'catalog_coverage@{K}'] = len(coverage) / core['catalog_size']
    summary[f'short_recall@{K}'] = df.loc[df['history_length'] <= 2, f'recall@{K}'].mean()
    summary[f'short_ndcg@{K}'] = df.loc[df['history_length'] <= 2, f'ndcg@{K}'].mean()
    return summary


def candidate_pool(user, n=80):
    components = build_components_for_user(user, n=200)
    base = tuned_recs(user, components, k=80)
    # Agregamos candidatos adicionales de componentes para que el reranking tenga espacio real de decision.
    pool = base + components[user]['seq'][:80] + components[user]['cat'][:80] + components[user]['hier'][:80]
    return filter_seen(pool, core['seen_items'].get(user, set()), n)

## 4. Comparacion de tradeoffs en muestra de usuarios

Evaluamos variantes de reranking en la misma muestra de 50.000 usuarios. La fila `tuned_base` corresponde al modelo final sin reranking adicional.

In [6]:
rerank_settings = [
    {'name': 'tuned_base', 'relevance_weight': 1.00, 'novelty_weight': 0.00, 'diversity_weight': 0.00},
    {'name': 'rerank_novelty_light', 'relevance_weight': 0.90, 'novelty_weight': 0.10, 'diversity_weight': 0.00},
    {'name': 'rerank_diversity_light', 'relevance_weight': 0.90, 'novelty_weight': 0.00, 'diversity_weight': 0.10},
    {'name': 'rerank_novelty_diversity_balanced', 'relevance_weight': 0.75, 'novelty_weight': 0.10, 'diversity_weight': 0.15},
    {'name': 'rerank_novelty_diversity_strong', 'relevance_weight': 0.60, 'novelty_weight': 0.20, 'diversity_weight': 0.20},
]

base_pools = {}
for idx, user in enumerate(core['eval_users'], start=1):
    base_pools[user] = candidate_pool(user, n=80)
    if idx % 10000 == 0:
        print(f'candidatos construidos para {idx:,} usuarios')

tradeoff_rows = []
for setting in rerank_settings:
    recs_by_user = {}
    for user, pool in base_pools.items():
        recs_by_user[user] = rerank_novelty_diversity(
            user,
            pool,
            k=K,
            relevance_weight=setting['relevance_weight'],
            novelty_weight=setting['novelty_weight'],
            diversity_weight=setting['diversity_weight'],
        )
    summary = evaluate_recs_for_users(recs_by_user)
    tradeoff_rows.append({'model': setting['name'], **summary})

tradeoffs = pd.DataFrame(tradeoff_rows).sort_values(f'recall@{K}', ascending=False)
tradeoffs.to_csv(OUTPUT_DIR / 'step4_reranking_tradeoffs_sample_50000.csv', index=False)
display(tradeoffs)

candidatos construidos para 10,000 usuarios


candidatos construidos para 20,000 usuarios


candidatos construidos para 30,000 usuarios


candidatos construidos para 40,000 usuarios


candidatos construidos para 50,000 usuarios


,model,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10,catalog_coverage@10,short_recall@10,short_ndcg@10
0,tuned_base,0.011320,0.11320,0.068543,13.598209,0.506618,0.299967,0.125735,0.076627
1,rerank_novelty_light,0.011320,0.11320,0.068543,13.598209,0.506618,0.299967,0.125735,0.076627
2,rerank_diversity_light,0.011320,0.11320,0.068543,13.598209,0.506618,0.299967,0.125735,0.076627
3,rerank_novelty_diversity_balanced,0.011320,0.11320,0.068543,13.598209,0.506618,0.299967,0.125735,0.076627
4,rerank_novelty_diversity_strong,0.011298,0.11298,0.068358,13.609523,0.510843,0.301651,0.125593,0.076451


## 5. Analisis por usuario

Buscamos usuarios donde el modelo tuneado acierta y donde el cambio respecto al hibrido fijo o al secuencial sea visible. Para cada caso mostramos historial, categorias, confianza, pesos adaptativos, recomendaciones y explicacion.

In [7]:
def explain_user(user):
    components = build_components_for_user(user, n=200)
    seq = components[user]['seq'][:K]
    cat = components[user]['cat'][:K]
    hier = components[user]['hier'][:K]
    fixed = fixed_hybrid_recs(user, components, K)
    tuned = tuned_recs(user, components, K)
    pool = candidate_pool(user, n=80)
    reranked = rerank_novelty_diversity(user, pool, K, relevance_weight=0.75, novelty_weight=0.10, diversity_weight=0.15)
    true_item = int(test_item[user])
    history, t_conf, h_len, weights = user_profile(user)
    last_categories = [h['categoryid'] for h in history if h['categoryid'] is not None]
    dominant_category = last_categories[-1] if last_categories else None
    return {
        'visitorid': user,
        'history_length': h_len,
        'transition_confidence': round(t_conf, 4),
        'adaptive_weights': weights,
        'recent_history': history,
        'last_known_category': dominant_category,
        'true_item': true_item,
        'true_category': core['item_to_category'].get(true_item),
        'sequential_top10': seq,
        'category_top10': cat,
        'hierarchical_top10': hier,
        'fixed_hybrid_top10': fixed,
        'tuned_top10': tuned,
        'reranked_top10': reranked,
        'sequential_hit': int(true_item in seq),
        'fixed_hit': int(true_item in fixed),
        'tuned_hit': int(true_item in tuned),
        'reranked_hit': int(true_item in reranked),
        'tuned_novelty': novelty_at_k(tuned, core['item_prob'], core['catalog_size'], K),
        'reranked_novelty': novelty_at_k(reranked, core['item_prob'], core['catalog_size'], K),
        'tuned_diversity': category_diversity_at_k(tuned, core['item_to_category'], K),
        'reranked_diversity': category_diversity_at_k(reranked, core['item_to_category'], K),
    }


selected_users = []
for user in core['eval_users']:
    info = explain_user(user)
    if info['history_length'] <= 2 and info['tuned_hit'] and len(selected_users) < 3:
        selected_users.append(info)
    elif info['history_length'] >= 5 and info['tuned_hit'] and len(selected_users) < 6:
        selected_users.append(info)
    if len(selected_users) >= 6:
        break

user_rows = []
for info in selected_users:
    user_rows.append({
        'visitorid': info['visitorid'],
        'history_length': info['history_length'],
        'transition_confidence': info['transition_confidence'],
        'weights': info['adaptive_weights'],
        'last_known_category': info['last_known_category'],
        'true_item': info['true_item'],
        'true_category': info['true_category'],
        'sequential_hit': info['sequential_hit'],
        'fixed_hit': info['fixed_hit'],
        'tuned_hit': info['tuned_hit'],
        'reranked_hit': info['reranked_hit'],
        'tuned_novelty': info['tuned_novelty'],
        'reranked_novelty': info['reranked_novelty'],
        'tuned_diversity': info['tuned_diversity'],
        'reranked_diversity': info['reranked_diversity'],
        'recent_history': info['recent_history'],
        'sequential_top10': info['sequential_top10'],
        'fixed_hybrid_top10': info['fixed_hybrid_top10'],
        'tuned_top10': info['tuned_top10'],
        'reranked_top10': info['reranked_top10'],
    })

user_analysis = pd.DataFrame(user_rows)
user_analysis.to_csv(OUTPUT_DIR / 'step4_user_level_analysis.csv', index=False)
display(user_analysis)

,visitorid,history_length,transition_confidence,weights,last_known_category,true_item,true_category,sequential_hit,fixed_hit,tuned_hit,reranked_hit,tuned_novelty,reranked_novelty,tuned_diversity,reranked_diversity,recent_history,sequential_top10,fixed_hybrid_top10,tuned_top10,reranked_top10
0,513229,1,0.8127,"{'seq': 0.5916461313980139, 'category': 0.3258562594057474, 'hierarchy': 0.03533560675397955, 'popularity': 0.047162002442258995}",1047,290993,484,1,1,1,1,14.170221,14.170221,0.666667,0.666667,"[{'event': 'view', 'itemid': 264876, 'categoryid': 1047, 'weight': 1.0}]","[126236, 340490, 290993, 406795, 463165, 20249, 108032, 233974, 366009, 56498]","[126236, 340490, 290993, 406795, 463165, 6692, 20249, 461686, 108032, 297513]","[126236, 340490, 290993, 6692, 406795, 463165, 297513, 20249, 461686, 109924]","[126236, 340490, 290993, 6692, 406795, 463165, 297513, 20249, 461686, 109924]"
1,1343397,2,0.2080,"{'seq': 0.33383769177282546, 'category': 0.46117158944231573, 'hierarchy': 0.155807995512595, 'popularity': 0.0491827232722638}",389,239766,996,1,1,1,1,15.215832,15.215832,0.666667,0.666667,"[{'event': 'view', 'itemid': 357329, 'categoryid': 996, 'weight': 1.0}, {'event': 'view', 'itemid': 155579, 'categoryid': 389, 'weight': 1.0}]","[239766, 461686, 257040, 119736, 320130, 9877, 309778, 312728, 384302, 7943]","[239766, 461686, 92214, 257040, 119736, 320130, 120644, 9877, 309778, 283699]","[92214, 239766, 283699, 120644, 461686, 51656, 257040, 200978, 119736, 52013]","[92214, 239766, 283699, 120644, 461686, 51656, 257040, 200978, 119736, 52013]"
2,486351,2,0.7920,"{'seq': 0.600894180468964, 'category': 0.31389492941066405, 'hierarchy': 0.03869987794267962, 'popularity': 0.04651101217769233}",1404,115764,1404,1,1,1,1,14.815943,14.815943,0.711111,0.711111,"[{'event': 'view', 'itemid': 345409, 'categoryid': 1373, 'weight': 1.0}, {'event': 'view', 'itemid': 140292, 'categoryid': 1404, 'weight': 1.0}]","[115764, 438122, 18573, 191223, 244116, 407809, 148042, 29757, 461686, 257040]","[115764, 438122, 158912, 18573, 191223, 82389, 244116, 461686, 407809, 148042]","[115764, 158912, 438122, 18573, 82389, 191223, 244116, 338148, 461686, 407809]","[115764, 158912, 438122, 18573, 82389, 191223, 244116, 338148, 461686, 407809]"
3,904138,8,1.0000,"{'seq': 0.7317203535827762, 'category': 0.22448267237777328, 'hierarchy': 0.0, 'popularity': 0.04379697403945048}",29,231243,29,1,1,1,1,14.679909,14.679909,0.377778,0.377778,"[{'event': 'view', 'itemid': 85307, 'categoryid': 29, 'weight': 1.0}, {'event': 'view', 'itemid': 432171, 'categoryid': 1554, 'weight': 1.0}, {'event': 'view', 'itemid': 432171...","[231243, 213993, 465833, 277137, 457189, 379520, 293864, 395876, 121757, 78978]","[231243, 379520, 213993, 465833, 277137, 254432, 457189, 395876, 293864, 371618]","[231243, 213993, 379520, 465833, 277137, 457189, 254432, 293864, 395876, 371618]","[231243, 213993, 379520, 465833, 277137, 457189, 254432, 293864, 395876, 371618]"
4,334660,12,0.4830,"{'seq': 0.526789228209947, 'category': 0.33353938601528205, 'hierarchy': 0.09414646267318012, 'popularity': 0.045524923101590876}",1219,345171,1219,1,1,1,1,12.343641,12.343641,0.533333,0.533333,"[{'event': 'view', 'itemid': 417187, 'categoryid': 1219, 'weight': 1.0}, {'event': 'view', 'itemid': 428648, 'categoryid': 808, 'weight': 1.0}, {'event': 'view', 'itemid': 4341...","[345171, 26087, 406296, 461686, 257040, 119736, 320130, 9877, 309778, 312728]","[345171, 26087, 48030, 406296, 461686, 257040, 464731, 119736, 320130, 27248]","[345171, 48030, 26087, 464731, 406296, 461686, 27248, 257040, 276704, 119736]","[345171, 48030, 26087, 464731, 406296, 461686, 27248, 257040, 276704, 119736]"
5,359293,5,0.4830,"{'seq': 0.49797330543859336, 'category': 0.3590649846996723, 'hierarchy': 0.09636432835925185, 'popularity': 0.046597381502482396}",34,340375,34,0,1,1,1,11.425132,11.425132,0.666667,0.666667,"[{'event': 'view', 'itemid': 3179, 'categoryid': 34, 'weight': 1.0}, {'event': 'view', 'itemid': 195065, 'cate

## 6. Lectura experimental

- El modelo final fue elegido para mejorar Recall@10 y nDCG@10, no para maximizar diversidad/novedad.
- Aun asi, la version tuneada mejora la novedad frente al hibrido fijo, porque incorpora mas senal jerarquica/categorial y no queda tan concentrada en transiciones directas.
- Si se quiere promover novedad/diversidad explicitamente, el reranking post-hoc es una alternativa simple. El resultado debe leerse como tradeoff: puede mejorar novelty/diversity, pero puede mover el item correcto a posiciones inferiores o sacarlo del top-10.
- Para el paper, esto permite decir que novedad/diversidad fueron evaluadas y que tambien se exploro una variante que las promueve explicitamente, aunque el modelo principal se selecciono por relevancia top-k.